In [ ]:
import logging
from loguru import logger
import sys
# Show only ERROR and above this notebook session
logging.basicConfig(level=logging.ERROR)
logger.remove()
logger.add(sys.stderr, level="ERROR")



Problem (training_together): Put it together (4 points)

Deliverable: Write a script that runs a training loop to train your model on user-provided input.

In particular, we recommend that your training script allow for (at least) the following:

- Ability to configure and control the various model and optimizer hyperparameters.
- Memory-eﬀicient loading of training and validation large datasets with np.memmap.
- Serializing checkpoints to a user-provided path.
- Periodically logging training and validation performance (e.g., to console and/or an external service like Weights and Biases)
- Every few iterations of training, output the predicted text (and decode it)


TODO: the tiktoken is currently set with a 50k vocab size ... we should set up a 10k vocab size as this will speed things
up.  See ron_bpe.py for example of 10k tokenizer.

We have a transformer language model that we will use a couple ways here.
1. We will trian it by feeding to batches of our trianing corpus, and evaluating the 
quality of the predictions
2. Every N iterations, we will provide it a "test-prompt" and let it tell us a little story.

prompt = tokenize("The ")
story = [prompt[0],]

for i in range(num_tok_stroy):
  output = ltm.forwrd(prompt)
  next_word = get_next_word(output) # 
  prompt = tokenize(next_word)
  story.append(next_word)


## Model and Optimizer parameter specification

In [ ]:
import numpy as np
import pathlib
import torch
from torchview import draw_graph
from cs336_basics.slamkin_bpe import convert_text_to_bpe
from cs336_basics.train_bpe import DEFAULT_SPECIAL_TOKENS
import tiktoken
from cs336_basics.modules import AdamW
from cs336_basics.modules import TransformerLanguageModel
from cs336_basics.modules import cross_entropy
from cs336_basics.modules import get_batch
from cs336_basics.modules import save_checkpoint
from cs336_basics.modules import load_checkpoint


encoding_name = "r50k_base"
tokenizer = tiktoken.get_encoding(encoding_name)

# Example of basic tokenization and detokenization
# output = tokenizer.encode("Hello world!", allowed_special={'<|endoftext|>'})
# print(output)
# round_trip = tokenizer.decode(output)
# print(round_trip)

In [ ]:
test_data = pathlib.Path("../data/test_timely_story.txt")
assert test_data.exists(), "Test data file does not exist. Please check the path."

# test_data = pathlib.Path("../data/TinyStoriesV2-GPT4-train.txt")
# assert test_data.exists(), "Test data file does not exist. Please check the path."

# Define the main parameters

In [ ]:
vocab_size = tokenizer.n_vocab  # 50257  # 10000  #50257,  # 50257 is the vocab size of the r50k_base tokenizer
context_length = 256  # 1024
num_layers = 4  # 48
d_model = 512  # 1600
num_heads = 16  #25
d_ff = 1344  # 6400
rope_theta = 10000

learning_rate = 1e-3 
learning_rate_warmup = 10  # 1000
adamw_betas = (0.9, 0.999)
adamw_eps = 1e-8
weight_decay = 0.01

num_batches_for_training = 4

token_dtype = np.uint16

# training_data_path = "data/processed/train.bin"


In [ ]:
# This takes 7 minutes to run on my machine, which is a bit long for a test, 

# n = convert_text_to_bpe(
#     test_data, 
#     special_tokens=DEFAULT_SPECIAL_TOKENS,
#     output_path=f"data/processed/tiny_stories_with_{encoding_name}.memmap",
#     dtype=token_dtype
#     )

In [ ]:
tokenized_training_data = "tiny_stories_tokenized_with_r50k_base_tiktoken.memmap"
# print(f"encoded with {n} tokens")
tokenized_text = np.memmap(tokenized_training_data, dtype=token_dtype) # , dtype=np.int32)
print(f"output shape: {tokenized_text.shape}, dtype: {tokenized_text.dtype}, first 100 tokens: {tokenized_text[:100]}")


In [ ]:
# len(np.unique(tokenized_text[:100]))

In [ ]:

for i in range(11):
    print(f"{tokenized_text[i]}: '{tokenizer.decode([tokenized_text[i]])}'")

print(tokenizer.decode(tokenized_text[:11]))

## Initialize the Transformer Language Model

In [ ]:
rope_params = {
    'theta': rope_theta,
    "max_seq_len": context_length
}   
tlm = TransformerLanguageModel(
    vocab_size=vocab_size,
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    rope_params=rope_params, 
    max_seq_len=context_length, # should be redundant with rope_params
)

In [ ]:
test_prompt = "What are you doing today?"

In the cell the below we look at how the next word selection is done on the logits.

Its all the same (and sounds childish) if we use:
`next_token_id = logits[0, -1, :].argmax().item()`
or 
`next_token_id = torch.softmax(logits[0, -1, :], dim=-1).argmax().item()`

we get much better performance when we use 
`next_token_id = torch.multinomial(torch.softmax(logits[0, -1, :], dim=-1), num_samples=1).item()`

The multinomial applied to 
`logits = torch.tensor([1, 0.0, -1]) ` has a 60% change of getting the 1, 24% 0, the 0 and around 9% -1.

However, say we use:
`logits = torch.tensor([1, 0.0, -1 ,-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])`
1 is still most likely ( around 25% of the time, but -1, has many instances ... and altogether, geting a -1-valued next word is like 65%)
This is consistent wtih the idea of many adjectives correspindng to the -1s, they are all similarly likely, but its quite likely that
an adjective is coming, but which adjective is a crapshoot.

In [ ]:
# logits = torch.tensor([1, 0.0, -1]) # tends to get 1 sampled more often than 0, and 0 more often than -1, but there is still some randomness
# logits = torch.tensor([1, 0.0, -1 ,-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])
logits = torch.tensor([3, 0.0, -1 ,-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])
print(f"Logits: {logits}")
probabilities = torch.softmax(logits, dim=-1)
print(f"Probabilities: {probabilities}")

for i in range(40):
    sample = torch.multinomial(probabilities, num_samples=1)
    print(f"Sample {i+1}: {sample.item()}, logit: {logits[sample.item()].item()}, probability: {probabilities[sample.item()].item():.4f}")

# torch.histogram(probabilities, bins=10).plot()


In [ ]:
def tell_me_a_little_story(prompt, tlm, tokenizer, max_length=50):
    input_ids = tokenizer.encode(prompt, allowed_special={'<|endoftext|>'})
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)  # shape (1, seq_len)
    
    for _ in range(max_length):
        with torch.no_grad():
            logits = tlm.forward(input_ids)

        # next_token_id = logits[0, -1, :].argmax().item()
        # next_token_id = torch.softmax(logits[0, -1, :], dim=-1).argmax().item()
        next_token_id = torch.multinomial(torch.softmax(logits[0, -1, :], dim=-1), num_samples=1).item()
        input_ids = torch.cat([input_ids, torch.tensor([[next_token_id]], dtype=torch.long)], dim=1)
        
        #if next_token_id == tokenizer.eot_token_id:
        #    break
    
    story = tokenizer.decode(input_ids[0].tolist())
    print("\n".join(story[i:i+120] for i in range(0, len(story), 120)))
    return story

In [ ]:

tell_me_a_little_story(test_prompt, tlm, tokenizer)
# tell_me_a_little_story("2+2=", tlm, tokenizer)

## Load best model to date

In [ ]:
# TODO warm up adam: warmup_steps=learning_rate_warmup,
adamw_optimizer = AdamW(
    params=tlm.parameters(),
    lr=learning_rate,
    betas=adamw_betas,
    eps=adamw_eps,
    weight_decay=weight_decay,
)
adamw_optimizer.zero_grad()  # zero out gradients before training loop

checkpoint_fname = f"checkpoint_step_911_loss.pt"
checkpoint_exists = pathlib.Path(checkpoint_fname).exists()
#use_checkpoint_iter=276
if checkpoint_exists:
    start_iteration = load_checkpoint(checkpoint_fname, tlm, adamw_optimizer)
    initial_loss = 2.84
    print(f"Resuming training from iteration {start_iteration} with initial loss {initial_loss:.4f}")
else:
    start_iteration = 0
    initial_loss = 100.0


In [ ]:
tell_me_a_little_story(test_prompt, tlm, tokenizer, max_length=context_length)
# tell_me_a_little_story("why not?", tlm, tokenizer, max_length=context_length)

## resume training loop

In [ ]:

num_training_steps = 128001
next_story_step = 1 # 2**torch.arange(10)


for i_step in range(start_iteration, num_training_steps):
        
    sampled_input_sequences, targets = get_batch(
        tokenized_text,
        batch_size=num_batches_for_training,
        context_length=context_length,
        device="cpu",
        )
    logits = tlm.forward(
        sampled_input_sequences,
        )
    # Print the 5 most probable

    # print(f"Logits shape: {logits.shape}, Targets shape: {targets.shape}")
    #next_tokens = logits[:, -1, :].argmax().item()
    #print(f"next_tokens: {next_tokens}, logits shape: {logits.shape}, targets shape: {targets.shape}")
    # gpt fix for cross_entropy input types
    if not isinstance(targets, torch.Tensor):
        targets = torch.tensor(targets, dtype=torch.long)
    elif targets.dtype != torch.long:
        targets = targets.long()
    
    mean_loss = cross_entropy(logits, targets)

    if mean_loss.item() < initial_loss:
        initial_loss = mean_loss.item()
        print(f"New lowest loss at step {i_step}: {initial_loss:.4f}")
        save_checkpoint(
            model=tlm, 
            optimizer=adamw_optimizer, 
            iteration=i_step, 
            out=f"checkpoint_step_{i_step}_loss.pt"
            ) 
        tell_me_a_little_story(test_prompt, tlm, tokenizer, max_length=context_length)


    adamw_optimizer.zero_grad()
    # TODO: backpropagate to update the model parameters using optimizers like AdamW
    mean_loss.backward()  # calculates gradients and stores them in the .grad attribute of each parameter
    # This shows what grad does: https://colab.research.google.com/drive/1OhZJs-zXCevwwvXYtbnqyy6mP-QIS_uk?hl=en#scrollTo=KgijKIe6dzpzcolab.research.google.comGoogle Colab
    adamw_optimizer.step()  # update
    
    if i_step == next_story_step:
        print(f"\n--- Story at step {i_step} --- (mean loss: {mean_loss.item():.4f})")
        tell_me_a_little_story(test_prompt, tlm, tokenizer)
        next_story_step *= 2



In [ ]:
batch_size = 2
tmp = draw_graph(
    tlm, 
    input_data=torch.randint(0, vocab_size, (batch_size, context_length)),
    depth=2, # how many layers of the model to show in the graph. 2 is enough to see the attention and feedforward submodules, without making the graph too cluttered   
    expand_nested=True, # whether to expand nested modules like the attention and feedforward submodules into their own nodes in the graph. This can make the graph more detailed but also more cluttered, so we set it to True here since we are only showing 2 layers
    hide_inner_tensors=True, # whether to hide inner tensors in the graph. This can make the graph less cluttered and easier to read, especially for large models, so we set it to True here
    roll=True, # whether to roll the graph to make it more compact. This can make the graph easier to read and fit on the screen, especially for large models, so we set it to True here
    hide_module_functions=False
    )
tmp.visual_graph

In [ ]:
#mean_loss.backward?

Debugging Stuff

In [ ]:
# sampled_input_sequences = [tokenizer.encode("The cat sat on the mat. The dog", allowed_special={'<|endoftext|>'})]
# sampled_input_sequences
# print(tokenizer.decode(sampled_input_sequences[0]))

In [ ]:
# # we use these for training
# sampled_input_sequences, predicted = get_batch(
#     tokenized_text, 
#     batch_size=num_batches_for_training, 
#     context_length=context_length, 
#     device="cpu",
#     )

In [ ]:
# print(sampled_input_sequences.shape)
# #sampled_input_subsequences[0:2, :5]
# sampled_input_subsequences = sampled_input_sequences[0:1, :11]
# tokenizer.decode(sampled_input_subsequences[0].tolist())

# i_tok = 33
# for i in range(4):
    
#     print(tokenizer.decode(sampled_input_sequences[i, :33].tolist()),"\n")

In [ ]:
# logits = tlm.forward(
#     sampled_input_sequences,
#     )
# logits[0, -1, 0:30]
# print(f" logits.shape: {logits.shape}"  ,"(batch, context_length, vocab_size)")

# # next word for the first sequence in the batch, based on the last position in the context
# i_batch = 0
# # logits[i_batch, -1, :].shape
# next_token = logits[i_batch, -1, :].argmax().item()
# print(f"Next token ID: {next_token}, which decodes to: '{tokenizer.decode([next_token])}'")

# here are the probabilities of the next word being each of the first 30 tokens in the vocabulary
# torch.softmax(logits[0, -1, :], dim=-1)[:30]
# TODO: study this later